In [2]:
import pandas as pd
import json
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer

In [2]:
df = pd.read_csv("data/healthcare-dataset-stroke-data.csv")

### One hot encoding for categorical variables 

In [3]:
# get list of all column names with categorical values
categorical_cols = ["gender", "ever_married", "work_type", "Residence_type", "smoking_status"]

# make a mapping of column names and the categories
categories = {i: [elem for elem in df[i].unique()] for i in categorical_cols }

# make a mapping of column names and the mapping to numerical values
category_to_encodings = {category: {categories[category][i]: i for i in range(len(categories[category]))} for category in categories}

# save our encodings in a json file for future reference
with open("data/encodings.json", "w") as jf:
    json.dump(category_to_encodings, jf, indent=4)

# use the encodings to replace values in the dataframe
for category in categories:
    df[category] = df[category].replace(category_to_encodings[category])

# save the encoded data (without NaN handling!) to a csv 
df.to_csv("data/encoded-stroke-data.csv", index=False)

/var/folders/yj/l0th9vq925q82mtdb1qykqpw0000gn/T/ipykernel_32780/2322765177.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[category] = df[category].replace(category_to_encodings[category])


### Handling the NaN values

2 options: 

- drop rows with NaN values 
- data imputation 

#### Dropping rows with NaN values

In [4]:
# drop rows with nan
df_drop = df.dropna()
print("Removed", df.shape[0]-df_drop.shape[0], "rows with NaN values in BMI.")

# save the version of the dataset with categorical encodings and dropped NaN rows 
df_drop.to_csv("data/dropped-stroke-data.csv", index=False)

Removed 201 rows with NaN values in BMI.


#### Data Imputation

In [5]:
# get encoded pandas dataframe as numpy array for imputation
df = pd.read_csv("data/encoded-stroke-data.csv")
np_data = df.to_numpy()
print("Shape of encoded data: ", np_data.shape)

Shape of encoded data:  (5110, 12)


##### Mean strategy for simple imputation 

In [6]:
# statistical imputation method - mean 
mean_imp = SimpleImputer(missing_values = np.nan, strategy='mean')
mean_imp.fit(np_data)
mean_np_data = mean_imp.transform(np_data)
mean_df = pd.DataFrame(mean_np_data)

# get cols and index pairs 
column_names_dict = {i: df.columns[i] for i in range(len(df.columns))}

# rename the columns 
mean_df = mean_df.rename(columns=column_names_dict)

# save the version of the dataset with categorical encodings and mean imputed NaN rows 
mean_df.to_csv("data/mean-stroke-data.csv", index=False)

##### Median strategy for simple imputation 

In [7]:
# statistical imputation method - median 
med_imp = SimpleImputer(missing_values = np.nan, strategy='median')
med_imp.fit(np_data)
med_np_data = med_imp.transform(np_data)
med_df = pd.DataFrame(med_np_data)

# get cols and index pairs 
column_names_dict = {i: df.columns[i] for i in range(len(df.columns))}

# rename the columns 
med_df = med_df.rename(columns=column_names_dict)

# save the version of the dataset with categorical encodings and median imputed NaN rows 
med_df.to_csv("data/median-stroke-data.csv", index=False)

##### KNN imputation

> **[Chanbin]**  
> *Open questions for the team.*
>
> **(1)** `KNNImputer` uses Euclidean distance on the numeric matrix as-is. Columns with a **wider numeric range** (e.g. `avg_glucose_level`, `age`) can **dominate** which rows count as nearest neighbors, while smaller-scale columns (BMI, 0/1 flags, low-cardinality encodings) may matter less in that distance. Is that a problem we should address here?
>
> **(2)** Should we scale or standardize features before KNN? Doing so can balance each feature’s influence on distance, but it also changes the neighbors and the imputed values. If we go that way, we should treat it as a **design choice**, not a single correct approach.

In [6]:
# read in encoded data
df = pd.read_csv("data/encoded-stroke-data.csv")

# drop and save id column
id_column = df["id"]
df = df.drop(["id"], axis=1)

# drop and save label column
label_column = df["stroke"]
df = df.drop(["stroke"], axis=1)

# transform into numpy array and check shape 
np_data = df.to_numpy()
print("Shape of encoded data with id col dropped: ", np_data.shape)

Shape of encoded data with id col dropped:  (5110, 10)


In [ ]:
# machine learning imputation method - KNN, K=5, uniform weights
knn_imp = KNNImputer(missing_values = np.nan, n_neighbors=5, weights="uniform")
knn_imp.fit(np_data)
knn_np_data = knn_imp.transform(np_data)
knn_df = pd.DataFrame(knn_np_data)

# get cols and index pairs 
column_names_dict = {i: df.columns[i] for i in range(len(df.columns))}

# rename the columns 
knn_df = knn_df.rename(columns=column_names_dict)

# reinsert the id and label column (excluded from fitting bc participant labels shouldn't impact nearest neighbors)
knn_df.insert(0, "id", id_column)
knn_df.insert(11, "stroke", label_column)

# save the version of the dataset with categorical encodings and KNN-imputed NaN rows (uniform weights)
knn_df.to_csv("data/knn-uniform-stroke-data.csv", index=False)

In [10]:
# machine learning imputation method - KNN, K=5, distance weights
knn_imp = KNNImputer(missing_values = np.nan, n_neighbors=5, weights="distance")
knn_imp.fit(np_data)
knn_np_data = knn_imp.transform(np_data)
knn_df = pd.DataFrame(knn_np_data)

# get cols and index pairs 
column_names_dict = {i: df.columns[i] for i in range(len(df.columns))}

# rename the columns 
knn_df = knn_df.rename(columns=column_names_dict)

# reinsert the id and label column (excluded from fitting bc participant labels shouldn't impact nearest neighbors)
knn_df.insert(0, "id", id_column)
knn_df.insert(11, "stroke", label_column)

# save the version of the dataset with categorical encodings and KNN-imputed NaN rows (distance weights)
knn_df.to_csv("data/knn-distance-stroke-data.csv", index=False)

KNN Imputation with StandardScaler

In [16]:
# machine learning imputation method - KNN, K=5, distance weights with standardization
from sklearn.preprocessing import StandardScaler

# standardize the data 
scaler = StandardScaler()
np_standardized = scaler.fit_transform(np_data)

# machine learning imputation method
knn_imp = KNNImputer(missing_values = np.nan, n_neighbors=5, weights="distance")
knn_imp.fit(np_standardized)
knn_np_data = knn_imp.transform(np_standardized)
knn_df = pd.DataFrame(knn_np_data)

# get cols and index pairs 
column_names_dict = {i: df.columns[i] for i in range(len(df.columns))}

# rename the columns 
knn_df = knn_df.rename(columns=column_names_dict)

# reinsert the id and label column (excluded from fitting bc participant labels shouldn't impact nearest neighbors)
knn_df.insert(0, "id", id_column)
knn_df.insert(11, "stroke", label_column)

# save the version of the dataset with categorical encodings and KNN-imputed NaN rows (distance weights)
knn_df.to_csv("data/knn-standardize-distance.csv", index=False)